# PII Metadata Pipeline — End-to-End Test
**Stack:** ydata-profiling + Microsoft Presidio + SQL Server

### Tables produced
| Table | Contents |
|---|---|
| `metadata_dataset_overview` | Row/col counts, duplicates, missing % |
| `metadata_column_stats` | Type, missing count/%, unique count/% |
| `metadata_descriptive_stats` | mean, std, min, max, quartiles, skewness, kurtosis |
| `metadata_correlations` | Pairwise Pearson correlations |
| `metadata_pii_column_summary` | Per-column PII flags, entity types, confidence scores |
| `metadata_pii_cell_detail` | Per-cell PII detections with confidence and char offsets |

## Cell 1 — Install dependencies
Run once, then **restart the kernel** before continuing.

In [ ]:
import sys

!{sys.executable} -m pip install -q ydata-profiling presidio-analyzer presidio-anonymizer pyodbc sqlalchemy faker
!{sys.executable} -m spacy download en_core_web_lg

print("\n✅ Installation complete — RESTART THE KERNEL now, then run from Cell 2.")

## Cell 2 — Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from datetime import datetime
from collections import defaultdict

from ydata_profiling import ProfileReport

from presidio_analyzer import AnalyzerEngine, RecognizerRegistry
from presidio_analyzer.nlp_engine import NlpEngineProvider
from presidio_analyzer.predefined_recognizers import (
    EmailRecognizer,
    PhoneRecognizer,
    CreditCardRecognizer,
    IpRecognizer,
    UsSsnRecognizer,
    DateRecognizer,
    SpacyRecognizer,
)

print("✅ Imports OK")

## Cell 3 — Verify spaCy model

In [ ]:
import spacy

installed = spacy.util.get_installed_models()
print("Installed spaCy models:", installed)

# Pick best available model
if "en_core_web_lg" in installed:
    SPACY_MODEL = "en_core_web_lg"
elif "en_core_web_md" in installed:
    SPACY_MODEL = "en_core_web_md"
elif "en_core_web_sm" in installed:
    SPACY_MODEL = "en_core_web_sm"
else:
    raise EnvironmentError("No spaCy English model found. Run Cell 1 and restart the kernel.")

print(f"✅ Using spaCy model: {SPACY_MODEL}")

## Cell 4 — Build Presidio analyzer with explicit recognizer registration

In [ ]:
# ── 1. NLP engine ─────────────────────────────────────────────────────────────
nlp_config = {
    "nlp_engine_name": "spacy",
    "models": [{"lang_code": "en", "model_name": SPACY_MODEL}],
}
nlp_engine = NlpEngineProvider(nlp_configuration=nlp_config).create_engine()

# ── 2. Registry — add each recognizer explicitly ───────────────────────────────
registry = RecognizerRegistry()

# Pattern-based (no NLP needed)
registry.add_recognizer(EmailRecognizer(supported_language="en"))
registry.add_recognizer(PhoneRecognizer(supported_language="en", supported_regions=["US", "GB", "CA"]))
registry.add_recognizer(CreditCardRecognizer(supported_language="en"))
registry.add_recognizer(IpRecognizer(supported_language="en"))
registry.add_recognizer(UsSsnRecognizer(supported_language="en"))
registry.add_recognizer(DateRecognizer(supported_language="en"))

# NLP-based (PERSON, LOCATION via spaCy NER)
registry.add_recognizer(SpacyRecognizer(
    supported_language="en",
    supported_entities=["PERSON", "LOCATION"],
    ner_strength=0.85,
))

# ── 3. Analyzer ───────────────────────────────────────────────────────────────
analyzer = AnalyzerEngine(
    nlp_engine=nlp_engine,
    registry=registry,
    supported_languages=["en"],
)

# ── 4. Verify ─────────────────────────────────────────────────────────────────
print("Registered recognizers:")
for r in analyzer.registry.recognizers:
    print(f"  {r.__class__.__name__:40s} → {r.supported_entities}")

# ── 5. Smoke test ─────────────────────────────────────────────────────────────
smoke_tests = [
    ("EMAIL",       "alice@example.com"),
    ("SSN",         "123-45-6789"),
    ("PHONE",       "+1-800-555-0199"),
    ("CREDIT CARD", "4111111111111111"),
    ("IP",          "192.168.1.1"),
    ("DATE",        "1990-04-15"),
]

print("\nSmoke test results:")
all_passed = True
for label, text in smoke_tests:
    results = analyzer.analyze(text=text, language="en")
    if results:
        best = max(results, key=lambda r: r.score)
        print(f"  ✅ [{label:12s}] '{text}' → {best.entity_type} (score={best.score:.2f})")
    else:
        print(f"  ❌ [{label:12s}] '{text}' → NO RESULT")
        all_passed = False

print("\n✅ All smoke tests passed!" if all_passed else "\n⚠️  Some tests failed — check recognizer list above.")

## Cell 5 — Generate synthetic test data

In [ ]:
from faker import Faker
import random

fake = Faker("en_US")
Faker.seed(42)
random.seed(42)
np.random.seed(42)

N = 50   # number of rows — increase for more thorough testing

def maybe_none(val, pct_missing=0.15):
    """Randomly replace a value with None to simulate missing data."""
    return None if random.random() < pct_missing else val

records = []
for _ in range(N):
    records.append({
        # PII columns
        "full_name":    maybe_none(fake.name()),
        "email":        maybe_none(fake.email()),
        "phone":        maybe_none(fake.phone_number()),
        "ssn":          maybe_none(fake.ssn()),                        # format: ###-##-####
        "credit_card":  maybe_none(fake.credit_card_number()),
        "dob":          maybe_none(str(fake.date_of_birth(minimum_age=18, maximum_age=70))),
        "ip_address":   maybe_none(fake.ipv4_private()),
        "address":      maybe_none(fake.address().replace("\n", ", ")),
        # Non-PII columns
        "department":   maybe_none(random.choice(["HR", "Engineering", "Finance", "Legal", "Marketing"])),
        "salary":       maybe_none(round(np.random.normal(75000, 15000), 2)),
        "tenure_years": maybe_none(round(random.uniform(0.5, 20.0), 1)),
        "performance":  maybe_none(round(np.random.uniform(1.0, 5.0), 1)),
    })

df = pd.DataFrame(records)

DATASET_NAME = "synthetic_employees"
RUN_TS       = datetime.utcnow()

print(f"Synthetic dataset: {df.shape[0]} rows × {df.shape[1]} cols")
print(f"\nMissing values per column:")
print(df.isna().sum())
df.head(5)

## Cell 6 — Run ydata-profiling

In [ ]:
profile = ProfileReport(
    df,
    title=DATASET_NAME,
    explorative=True,
    minimal=False,
)

# Optional: view the full report inline
# profile.to_notebook_iframe()

print("✅ Profiling complete")

## Cell 7 — Extract ydata-profiling metadata

In [ ]:
desc        = profile.get_description()
table_stats = desc.table
n_rows      = int(table_stats.get("n", 1))

# ── Dataset overview ──────────────────────────────────────────────────────────
overview_df = pd.DataFrame([{
    "dataset_name":       DATASET_NAME,
    "run_timestamp":      RUN_TS,
    "n_rows":             n_rows,
    "n_cols":             int(table_stats.get("n_var", 0)),
    "n_missing_cells":    int(table_stats.get("n_cells_missing", 0)),
    "pct_missing_cells":  round(float(table_stats.get("p_cells_missing", 0)) * 100, 4),
    "n_duplicate_rows":   int(table_stats.get("n_duplicates", 0)),
    "pct_duplicate_rows": round(float(table_stats.get("p_duplicates", 0)) * 100, 4),
    "n_numeric_cols":     int(table_stats.get("types", {}).get("Numeric", 0)),
    "n_categorical_cols": int(table_stats.get("types", {}).get("Categorical", 0)),
}])

# ── Column stats + descriptive stats ──────────────────────────────────────────
col_stats_rows, desc_stats_rows = [], []

for col_name, col_data in desc.variables.items():
    col_type = str(col_data.get("type", "Unknown"))

    col_stats_rows.append({
        "dataset_name":  DATASET_NAME,
        "run_timestamp": RUN_TS,
        "column_name":   col_name,
        "column_type":   col_type,
        "n_missing":     int(col_data.get("n_missing", 0)),
        "pct_missing":   round(float(col_data.get("p_missing", 0)) * 100, 4),
        "n_unique":      int(col_data.get("n_unique", col_data.get("n_distinct", 0))),
        "pct_unique":    round(int(col_data.get("n_unique", col_data.get("n_distinct", 0))) / n_rows * 100, 4),
    })

    if col_type in ("Numeric", "NUM") or col_data.get("mean") is not None:
        desc_stats_rows.append({
            "dataset_name":  DATASET_NAME,
            "run_timestamp": RUN_TS,
            "column_name":   col_name,
            "mean":          round(float(col_data.get("mean",     0) or 0), 6),
            "std":           round(float(col_data.get("std",      0) or 0), 6),
            "min":           round(float(col_data.get("min",      0) or 0), 6),
            "pct_25":        round(float(col_data.get("25%",      0) or 0), 6),
            "median":        round(float(col_data.get("50%",      0) or 0), 6),
            "pct_75":        round(float(col_data.get("75%",      0) or 0), 6),
            "max":           round(float(col_data.get("max",      0) or 0), 6),
            "kurtosis":      round(float(col_data.get("kurtosis", 0) or 0), 6),
            "skewness":      round(float(col_data.get("skewness", 0) or 0), 6),
        })

col_stats_df  = pd.DataFrame(col_stats_rows)
desc_stats_df = pd.DataFrame(desc_stats_rows)

# ── Correlations ──────────────────────────────────────────────────────────────
corr_rows = []
if desc.correlations:
    corr_key = "pearson" if "pearson" in desc.correlations else next(iter(desc.correlations), None)
    if corr_key:
        corr_matrix = desc.correlations[corr_key]
        for col_a in corr_matrix.columns:
            for col_b in corr_matrix.columns:
                if col_a < col_b:
                    val = corr_matrix.loc[col_a, col_b] if col_a in corr_matrix.index else None
                    if val is not None and not pd.isna(val):
                        corr_rows.append({
                            "dataset_name":      DATASET_NAME,
                            "run_timestamp":     RUN_TS,
                            "correlation_type":  corr_key,
                            "column_a":          col_a,
                            "column_b":          col_b,
                            "correlation_value": round(float(val), 6),
                        })

corr_df = pd.DataFrame(corr_rows)

print("✅ ydata-profiling metadata extracted")
print(f"   overview: {len(overview_df)} row | col_stats: {len(col_stats_df)} rows | "
      f"desc_stats: {len(desc_stats_df)} rows | correlations: {len(corr_df)} rows")
display(overview_df)
display(col_stats_df)

## Cell 8 — PII scanning with per-entity confidence thresholds

In [ ]:
# ── Per-entity thresholds ─────────────────────────────────────────────────────
# Pattern-based recognizers (EMAIL, SSN, IP, CC) score at fixed values
# NER-based recognizers (PERSON, LOCATION) return floating probabilities
ENTITY_THRESHOLDS = {
    "EMAIL_ADDRESS": 0.5,
    "US_SSN":        0.5,
    "PHONE_NUMBER":  0.5,
    "CREDIT_CARD":   0.5,
    "IP_ADDRESS":    0.5,
    "DATE_TIME":     0.6,
    "PERSON":        0.7,
    "LOCATION":      0.7,
}

PII_ENTITIES = list(ENTITY_THRESHOLDS.keys())

# ── Scan all string columns ───────────────────────────────────────────────────
pii_cell_rows   = []
pii_col_staging = defaultdict(list)

string_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
print(f"Scanning {len(string_cols)} string column(s): {string_cols}\n")

for col in string_cols:
    col_detections = 0
    for row_idx, cell_value in df[col].items():
        if pd.isna(cell_value) or str(cell_value).strip() == "":
            continue

        try:
            results = analyzer.analyze(
                text=str(cell_value),
                entities=PII_ENTITIES,
                language="en",
            )
        except Exception as e:
            print(f"  ⚠️  Error on col='{col}' row={row_idx}: {e}")
            continue

        for result in results:
            threshold = ENTITY_THRESHOLDS.get(result.entity_type, 0.5)
            if result.score < threshold:
                continue

            pii_cell_rows.append({
                "dataset_name":  DATASET_NAME,
                "run_timestamp": RUN_TS,
                "column_name":   col,
                "row_index":     int(row_idx),
                "cell_value":    str(cell_value),
                "entity_type":   result.entity_type,
                "confidence":    round(result.score, 6),
                "start_char":    result.start,
                "end_char":      result.end,
            })
            pii_col_staging[col].append((result.entity_type, result.score))
            col_detections += 1

    print(f"  {col:20s} → {col_detections} detection(s)")

pii_cell_df = pd.DataFrame(pii_cell_rows) if pii_cell_rows else pd.DataFrame(
    columns=["dataset_name","run_timestamp","column_name","row_index",
             "cell_value","entity_type","confidence","start_char","end_char"]
)

print(f"\n✅ Total cell-level PII detections: {len(pii_cell_df)}")
pii_cell_df.head(10)

## Cell 9 — Build per-column PII summary

In [ ]:
pii_col_rows = []

for col in string_cols:
    detections   = pii_col_staging.get(col, [])
    n_non_null   = int(df[col].notna().sum())
    entity_types = sorted({e for e, _ in detections})
    scores       = [s for _, s in detections]
    flagged_rows = (
        pii_cell_df[pii_cell_df["column_name"] == col]["row_index"].nunique()
        if not pii_cell_df.empty else 0
    )

    pii_col_rows.append({
        "dataset_name":           DATASET_NAME,
        "run_timestamp":          RUN_TS,
        "column_name":            col,
        "is_pii":                 len(entity_types) > 0,
        "detected_entity_types":  ", ".join(entity_types) if entity_types else None,
        "n_entity_types":         len(entity_types),
        "avg_confidence":         round(float(np.mean(scores)), 6)  if scores else None,
        "max_confidence":         round(float(np.max(scores)),  6)  if scores else None,
        "min_confidence":         round(float(np.min(scores)),  6)  if scores else None,
        "n_flagged_rows":         flagged_rows,
        "pct_flagged_rows":       round(flagged_rows / n_non_null * 100, 4) if n_non_null > 0 else 0.0,
        "thresholds_applied":     str(ENTITY_THRESHOLDS),
    })

pii_col_df = pd.DataFrame(pii_col_rows)

print("✅ Per-column PII summary:")
display(pii_col_df[["column_name","is_pii","detected_entity_types",
                     "n_flagged_rows","pct_flagged_rows","avg_confidence","max_confidence"]])

## Cell 10 — SQL Server connection
**Skip this cell if you only want local testing — jump to Cell 12 instead.**

In [ ]:
from sqlalchemy import create_engine, text
import urllib

# ── CONFIGURE THESE ───────────────────────────────────────────────────────────
SQL_SERVER   = "YOUR_SERVER_NAME"
SQL_DATABASE = "YOUR_DATABASE_NAME"
SQL_SCHEMA   = "dbo"

# Option A: Windows Authentication
params = urllib.parse.quote_plus(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={SQL_SERVER};DATABASE={SQL_DATABASE};"
    f"Trusted_Connection=yes;"
)

# Option B: Username & Password
# SQL_USERNAME = "YOUR_USERNAME"
# SQL_PASSWORD = "YOUR_PASSWORD"
# params = urllib.parse.quote_plus(
#     f"DRIVER={{ODBC Driver 17 for SQL Server}};"
#     f"SERVER={SQL_SERVER};DATABASE={SQL_DATABASE};"
#     f"UID={SQL_USERNAME};PWD={SQL_PASSWORD};"
# )

engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")
with engine.connect() as conn:
    conn.execute(text("SELECT 1"))
print("✅ SQL Server connection successful")

## Cell 11 — Auto-create SQL Server tables

In [ ]:
DDL_STATEMENTS = [
f"""IF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name='metadata_dataset_overview')
CREATE TABLE [{SQL_SCHEMA}].[metadata_dataset_overview] (
    id INT IDENTITY(1,1) PRIMARY KEY, dataset_name NVARCHAR(255) NOT NULL,
    run_timestamp DATETIME2 NOT NULL, n_rows INT, n_cols INT,
    n_missing_cells INT, pct_missing_cells FLOAT,
    n_duplicate_rows INT, pct_duplicate_rows FLOAT,
    n_numeric_cols INT, n_categorical_cols INT);""",

f"""IF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name='metadata_column_stats')
CREATE TABLE [{SQL_SCHEMA}].[metadata_column_stats] (
    id INT IDENTITY(1,1) PRIMARY KEY, dataset_name NVARCHAR(255) NOT NULL,
    run_timestamp DATETIME2 NOT NULL, column_name NVARCHAR(255) NOT NULL,
    column_type NVARCHAR(100), n_missing INT, pct_missing FLOAT,
    n_unique INT, pct_unique FLOAT);""",

f"""IF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name='metadata_descriptive_stats')
CREATE TABLE [{SQL_SCHEMA}].[metadata_descriptive_stats] (
    id INT IDENTITY(1,1) PRIMARY KEY, dataset_name NVARCHAR(255) NOT NULL,
    run_timestamp DATETIME2 NOT NULL, column_name NVARCHAR(255) NOT NULL,
    mean FLOAT, std FLOAT, min FLOAT, pct_25 FLOAT, median FLOAT,
    pct_75 FLOAT, max FLOAT, kurtosis FLOAT, skewness FLOAT);""",

f"""IF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name='metadata_correlations')
CREATE TABLE [{SQL_SCHEMA}].[metadata_correlations] (
    id INT IDENTITY(1,1) PRIMARY KEY, dataset_name NVARCHAR(255) NOT NULL,
    run_timestamp DATETIME2 NOT NULL, correlation_type NVARCHAR(50),
    column_a NVARCHAR(255) NOT NULL, column_b NVARCHAR(255) NOT NULL,
    correlation_value FLOAT);""",

f"""IF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name='metadata_pii_column_summary')
CREATE TABLE [{SQL_SCHEMA}].[metadata_pii_column_summary] (
    id INT IDENTITY(1,1) PRIMARY KEY, dataset_name NVARCHAR(255) NOT NULL,
    run_timestamp DATETIME2 NOT NULL, column_name NVARCHAR(255) NOT NULL,
    is_pii BIT NOT NULL, detected_entity_types NVARCHAR(500),
    n_entity_types INT, avg_confidence FLOAT, max_confidence FLOAT,
    min_confidence FLOAT, n_flagged_rows INT, pct_flagged_rows FLOAT,
    thresholds_applied NVARCHAR(MAX));""",

f"""IF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name='metadata_pii_cell_detail')
CREATE TABLE [{SQL_SCHEMA}].[metadata_pii_cell_detail] (
    id INT IDENTITY(1,1) PRIMARY KEY, dataset_name NVARCHAR(255) NOT NULL,
    run_timestamp DATETIME2 NOT NULL, column_name NVARCHAR(255) NOT NULL,
    row_index INT NOT NULL, cell_value NVARCHAR(MAX), entity_type NVARCHAR(100) NOT NULL,
    confidence FLOAT NOT NULL, start_char INT, end_char INT);"""
]

with engine.begin() as conn:
    for stmt in DDL_STATEMENTS:
        conn.execute(text(stmt))
print("✅ All 6 tables created (or already exist)")

## Cell 12 — Insert into SQL Server

In [ ]:
TABLE_MAP = {
    "metadata_dataset_overview":   overview_df,
    "metadata_column_stats":       col_stats_df,
    "metadata_descriptive_stats":  desc_stats_df,
    "metadata_correlations":       corr_df,
    "metadata_pii_column_summary": pii_col_df,
    "metadata_pii_cell_detail":    pii_cell_df,
}

with engine.begin() as conn:
    for table_name, df_to_insert in TABLE_MAP.items():
        if df_to_insert.empty:
            print(f"  ⚠️  Skipped {table_name} (empty)")
            continue
        df_to_insert.to_sql(
            name=table_name, con=conn, schema=SQL_SCHEMA,
            if_exists="append", index=False
        )
        print(f"  ✅ Inserted {len(df_to_insert):>4} row(s) → [{SQL_SCHEMA}].[{table_name}]")

print("\n✅ All metadata written to SQL Server")

## Cell 13 — Local results preview (no SQL Server needed)

In [ ]:
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
display(overview_df)

print("\n" + "=" * 60)
print("PII COLUMN SUMMARY")
print("=" * 60)
display(pii_col_df[[
    "column_name", "is_pii", "detected_entity_types",
    "n_flagged_rows", "pct_flagged_rows", "avg_confidence", "max_confidence"
]])

print("\n" + "=" * 60)
print("PII CELL DETAIL — first 20 rows")
print("=" * 60)
display(pii_cell_df[["column_name","row_index","cell_value",
                      "entity_type","confidence"]].head(20))

print("\n" + "=" * 60)
print("COLUMN STATS")
print("=" * 60)
display(col_stats_df)

print("\n" + "=" * 60)
print("DESCRIPTIVE STATS")
print("=" * 60)
display(desc_stats_df)

## Cell 14 — Useful SQL queries
```sql
-- All PII-positive columns
SELECT column_name, detected_entity_types, pct_flagged_rows, avg_confidence
FROM dbo.metadata_pii_column_summary
WHERE dataset_name = 'synthetic_employees' AND is_pii = 1
ORDER BY avg_confidence DESC;

-- High-confidence detections only
SELECT column_name, row_index, cell_value, entity_type, confidence
FROM dbo.metadata_pii_cell_detail
WHERE dataset_name = 'synthetic_employees' AND confidence >= 0.85
ORDER BY confidence DESC;

-- Full column inventory: profiling stats + PII flag
SELECT cs.column_name, cs.column_type, cs.pct_missing,
       pii.is_pii, pii.detected_entity_types, pii.avg_confidence
FROM dbo.metadata_column_stats cs
LEFT JOIN dbo.metadata_pii_column_summary pii
       ON cs.column_name   = pii.column_name
      AND cs.dataset_name  = pii.dataset_name
      AND cs.run_timestamp = pii.run_timestamp
WHERE cs.dataset_name = 'synthetic_employees';
```